# Hashing — Dict, Set & the `collections` Family — Interview Masterclass

The companion to the arrays notebook. Hashing is the **second most-used pattern** in array problems — InterviewBit's analysis says HashMap optimization alone covers ~8 of the 30 most-asked array questions. This notebook covers the data structures (`dict`, `set`, `Counter`, `defaultdict`, `OrderedDict`), the canonical patterns, and the interview talking points you need to sound senior.

Out of scope by design: binary search, sorting algorithms, recursion, backtracking, dynamic programming.

## How to use this notebook
1. Internalize **section 1** first — the conceptual model of hashing. Most candidates fumble these questions ("why O(1)?", "what makes a key hashable?", "what is a hash collision?").
2. Drill the method references (sections 2-4) until you don't have to think.
3. Pattern sections (5-13) — for each, read the concept, run the code, then practice the listed LeetCode problems.
4. Section 14 is the cheat sheet — keep it open during mock interviews.

---

# 1. Hashing — the conceptual model

## What is a hash table?
A hash table stores key-value pairs in an internal array. To find where a key goes, it computes `hash(key)` (a fixed-size integer) and uses `hash(key) % capacity` as the index. Lookup, insert, and delete are all **O(1) on average**. The price you pay is no order guarantee in classical implementations and worst-case O(n) when too many keys hit the same bucket (collision).

## What is a collision?
Two different keys producing the same bucket index. There are two main strategies to handle them:
- **Chaining** — each bucket holds a linked list (or, in modern Java 8+, a balanced tree if the list gets long). Python uses a variant of this.
- **Open addressing** — on collision, probe the next slot (linear, quadratic, or double hashing). C++ `std::unordered_map` uses open addressing.

## Why does Python's dict claim O(1)?
Because the load factor is kept low. CPython resizes the table once it's ~2/3 full, which keeps collisions rare and lookups effectively O(1). If you ever see "O(1) amortized" in writing, this is what's being amortized.

## CPython's compact dict (Python 3.6+)
A modern CPython `dict` is implemented as **two arrays**:
1. A **sparse index array** (~33% full) for the hash table — stores indices into the entries array.
2. A **dense entries array** that stores `(hash, key, value)` tuples in **insertion order**.

This compact layout was introduced in Python 3.6 (as an implementation detail) and **made a language guarantee in Python 3.7+**. Two important consequences:
- **Insertion order is preserved.** Iterating a dict gives you keys in the order they were first added.
- **`dict` uses less memory** than a textbook hash table because the sparse index only stores small ints, not full entries.

## What makes a Python object usable as a key?
It must be **hashable** — i.e. its `__hash__` returns a stable int, and its `__eq__` is consistent (two equal objects must hash the same).

| Type | Hashable? | Why |
|---|---|---|
| `int`, `float`, `str`, `bool`, `bytes` | Yes | Immutable |
| `tuple` (of hashables) | Yes | Immutable, recursive check |
| `frozenset` | Yes | Immutable set |
| `None` | Yes | Singleton |
| `list` | **No** | Mutable |
| `dict` | **No** | Mutable |
| `set` | **No** | Mutable |
| Custom class | Default yes | Hashable by `id()` unless you override `__eq__` without `__hash__` |

The rule: **if the object could be mutated to no longer equal itself, it cannot be a key.** A list could be extended; if it were a key, the table couldn't find it again.

## When to reach for a hash table in an interview
- "Have I seen X before?" → set
- "How many times does X appear?" → dict / Counter
- "Map A to B" → dict
- "Brute force has a nested loop where inner is 'find X'" → dict eliminates the inner loop
- "Order matters AND I need O(1) lookups" → ordered dict (regular dict in 3.7+)


---
# 2. `dict` — complete reference

Operating-complexity table:

| Operation | Average | Worst | Notes |
|---|---|---|---|
| `d[key]`, `d[key] = v` | O(1) | O(n) | Worst case is pathological collisions |
| `key in d` | O(1) | O(n) | |
| `del d[key]` | O(1) | O(n) | |
| `d.get(key, default)` | O(1) | O(n) | No KeyError on miss |
| `d.setdefault(key, default)` | O(1) | O(n) | Inserts default if missing |
| `d.pop(key, default)` | O(1) | O(n) | |
| `d.popitem()` | O(1) | O(1) | Removes & returns LAST inserted pair (LIFO) |
| Iteration | O(n) | O(n) | Insertion order |
| `len(d)` | O(1) | O(1) | |
| Shallow copy `d.copy()` | O(n) | O(n) | |

In [ ]:
# 2.1 Construction — five ways
d1 = {'a': 1, 'b': 2}                          # literal
d2 = dict(a=1, b=2)                            # keyword args
d3 = dict([('a', 1), ('b', 2)])                # from list of pairs
d4 = dict(zip(['a', 'b'], [1, 2]))             # from two parallel sequences
d5 = {k: k*k for k in range(5)}                # comprehension
d6 = dict.fromkeys(['x', 'y', 'z'], 0)         # all keys -> same default value

print(d1, d2, d3, d4, d5, d6)


In [ ]:
# 2.2 Adding / updating
d = {'a': 1}

d['b'] = 2                          # add or overwrite
d.update({'c': 3, 'd': 4})          # bulk merge — overwrites duplicates
d.update(e=5, f=6)                  # kwargs form
print(d)

# Python 3.9+ merge operators
a = {'x': 1, 'y': 2}
b = {'y': 99, 'z': 3}
print(a | b)                        # NEW dict — b wins on conflicts
a |= b                              # in-place merge
print(a)


In [ ]:
# 2.3 Reading — get vs setdefault vs []
d = {'a': 1}

print(d['a'])                       # 1 — KeyError if missing
print(d.get('z'))                   # None — never raises
print(d.get('z', -1))               # -1 — default

# setdefault — insert default IF missing, return current value
print(d.setdefault('a', 999))       # 1   — already exists, returns 1
print(d.setdefault('z', 42))        # 42  — inserts z=42
print(d)                            # {'a': 1, 'z': 42}

# Counting idiom WITHOUT defaultdict (slower but works everywhere)
counts = {}
for ch in "hello":
    counts[ch] = counts.get(ch, 0) + 1
print(counts)


In [ ]:
# 2.4 Removing
d = {'a': 1, 'b': 2, 'c': 3, 'd': 4}

print(d.pop('a'))                   # 1, removed     — KeyError if missing
print(d.pop('z', None))             # None — safe pop with default
print(d.popitem())                  # ('d', 4) — removes LAST inserted (LIFO since 3.7)

del d['b']                          # KeyError if missing
print(d)

d.clear()
print(d)


In [ ]:
# 2.5 Views — keys/values/items are LIVE views, not copies
d = {'a': 1, 'b': 2, 'c': 3}

ks = d.keys()
vs = d.values()
it = d.items()

print(list(ks), list(vs), list(it))

# Mutating the dict updates the views in real time
d['z'] = 99
print(list(ks))                     # ['a','b','c','z']

# Set-like operations on key views (because keys are unique like a set)
d1 = {'a': 1, 'b': 2, 'c': 3}
d2 = {'b': 20, 'c': 30, 'd': 40}
print(d1.keys() & d2.keys())        # {'b', 'c'} — common keys
print(d1.keys() | d2.keys())        # all keys
print(d1.keys() - d2.keys())        # {'a'}


In [1]:
# 2.6 Iteration patterns
d = {'a': 1, 'b': 2, 'c': 3}

for k in d:                         # iterates KEYS (not items!)
    print(k)

for v in d.values():
    print(v)

for k, v in d.items():              # the one you want 90% of the time
    print(k, v)

# Sort by value (returns a list of pairs; not a dict)
sorted_by_val = sorted(d.items(), key=lambda kv: kv[1], reverse=True)
print(sorted_by_val)

# Convert back to a dict if you need to (insertion order = sorted order in 3.7+)
print(dict(sorted_by_val))

# Find key with max value
print(max(d, key=d.get))            # the key whose value is largest


a
b
c
1
2
3
a 1
b 2
c 3
[('c', 3), ('b', 2), ('a', 1)]
{'c': 3, 'b': 2, 'a': 1}
c


In [ ]:
# 2.7 Reversing a dict — keys ↔ values
d = {'a': 1, 'b': 2, 'c': 3}
inv = {v: k for k, v in d.items()}
print(inv)                          # {1:'a', 2:'b', 3:'c'}

# CAUTION: if multiple keys share the same value, inversion is lossy.
# To keep ALL keys per value, group instead:
d2 = {'a': 1, 'b': 2, 'c': 1}
from collections import defaultdict
groups = defaultdict(list)
for k, v in d2.items():
    groups[v].append(k)
print(dict(groups))                 # {1: ['a','c'], 2:['b']}


In [ ]:
# 2.8 Copying — same shallow/deep distinction as lists
import copy

d = {'inner': [1, 2, 3]}
shallow = d.copy()                  # outer is new; inner list is SHARED
deep = copy.deepcopy(d)             # fully independent

d['inner'].append(99)
print("d:      ", d)
print("shallow:", shallow)          # also has 99 — aliased
print("deep:   ", deep)             # unchanged


---
# 3. `set` and `frozenset` — complete reference

A `set` is an unordered collection of **unique, hashable** elements. Internally it's basically a dict with no values — same hash table, same O(1) average ops.

`frozenset` is the immutable, hashable cousin. Use it when you need a set **as a dict key** or **inside another set**.

| Operation | Average | Worst |
|---|---|---|
| `x in s` | O(1) | O(n) |
| `s.add(x)`, `s.discard(x)` | O(1) | O(n) |
| `s | t` (union), `s & t` (intersect) | O(len(s) + len(t)) | |
| `s - t` (difference) | O(len(s)) | |
| `s <= t` (subset) | O(len(s)) | |
| Iteration | O(n) | |

In [ ]:
# 3.1 Construction
a = {1, 2, 3}                       # literal
b = set([1, 2, 3, 3])               # from iterable — dedups
c = set("hello")                    # {'h','e','l','o'}
d = {x*x for x in range(5)}         # set comprehension
e = set()                           # EMPTY set — {} would be an empty DICT
fs = frozenset([1, 2, 3])           # immutable, hashable

print(a, b, c, d, e, fs)


In [ ]:
# 3.2 Adding / removing
s = {1, 2, 3}

s.add(4)                            # add single element
s.update([5, 6], {7})               # add multiple iterables at once
print(s)

s.remove(2)                         # KeyError if missing
s.discard(99)                       # no error if missing  — prefer this in defensive code
print(s.pop())                      # arbitrary element — remember sets are unordered
print(s)
s.clear()


In [ ]:
# 3.3 Set algebra — interview gold
a = {1, 2, 3, 4}
b = {3, 4, 5, 6}

print(a | b)                        # union           {1,2,3,4,5,6}
print(a & b)                        # intersection    {3,4}
print(a - b)                        # difference      {1,2}    (in a, not in b)
print(b - a)                        # difference      {5,6}
print(a ^ b)                        # symmetric diff  {1,2,5,6}  — in exactly one

# Method forms (work with any iterable, not just a set)
print(a.union([3, 4, 5]))
print(a.intersection([3, 4, 5]))

# In-place variants
a |= {99}                           # a.update
a &= {1, 99}                        # a.intersection_update
a ^= {1}                            # a.symmetric_difference_update
print(a)


In [ ]:
# 3.4 Subset / superset / disjoint
a = {1, 2}
b = {1, 2, 3, 4}

print(a <= b)                       # subset             — also a.issubset(b)
print(a < b)                        # proper subset      (a != b)
print(b >= a)                       # superset           — also b.issuperset(a)
print({1,2}.isdisjoint({3,4}))      # True — no common


---
# 4. The `collections` family — three you must know cold

Knowing these signals seniority to an interviewer. **Frequency counting** is the most common dict use case in interviews, and `Counter` is the idiomatic Pythonic tool.

In [ ]:
# 4.1 Counter — frequency counting on steroids
from collections import Counter

c = Counter("mississippi")
print(c)                            # Counter({'i':4,'s':4,'p':2,'m':1})
print(c['i'])                       # 4
print(c['z'])                       # 0   — missing key returns 0, NOT KeyError

c.update("missile")                 # accumulate counts
print(c)

# most_common — top-K frequent
print(c.most_common(2))             # [('i', 5), ('s', 5)]  — list of pairs

# Arithmetic between counters
c1 = Counter("aab")
c2 = Counter("abb")
print(c1 + c2)                      # Counter({'a':3,'b':3})
print(c1 - c2)                      # Counter({'a':1})       — keeps only positives
print(c1 & c2)                      # Counter({'a':1,'b':1}) — intersection (min)
print(c1 | c2)                      # Counter({'a':2,'b':2}) — union (max)

# Convert two-way: list of elements <-> counter
print(list(c1.elements()))          # ['a','a','b']
print(Counter(['a','a','b']))


In [ ]:
# 4.2 defaultdict — autocreate missing keys
from collections import defaultdict

# Group strings by first letter
words = ['apple', 'banana', 'avocado', 'blueberry', 'cherry']
groups = defaultdict(list)
for w in words:
    groups[w[0]].append(w)
print(dict(groups))                 # {'a':['apple','avocado'], 'b':[...], 'c':[...]}

# Counting with defaultdict(int) — alternative to Counter
counts = defaultdict(int)
for ch in "mississippi":
    counts[ch] += 1
print(dict(counts))

# Nested defaultdict — common in interview design problems
# date -> department -> total
nested = defaultdict(lambda: defaultdict(int))
nested['2026-03-01']['eng'] += 500
nested['2026-03-01']['mkt'] += 300
nested['2026-03-02']['eng'] += 700
print({k: dict(v) for k, v in nested.items()})

# WARNING: accessing a missing key CREATES it
dd = defaultdict(list)
print(dd['ghost'])                  # []
print(dict(dd))                     # {'ghost': []}   <-- key now exists!


In [ ]:
# 4.3 OrderedDict — when regular dict isn't enough
# Since Python 3.7+, regular dict preserves insertion order — so why OrderedDict?
# Two reasons:
#   1. Order-sensitive equality
#   2. move_to_end() — essential for LRU cache

from collections import OrderedDict

# 1. Order-sensitive equality
od1 = OrderedDict([('a', 1), ('b', 2)])
od2 = OrderedDict([('b', 2), ('a', 1)])
print(od1 == od2)                   # False — order matters!

r1 = {'a': 1, 'b': 2}
r2 = {'b': 2, 'a': 1}
print(r1 == r2)                     # True  — regular dict ignores order

# 2. move_to_end — used in LRU caches
od = OrderedDict([('a', 1), ('b', 2), ('c', 3)])
od.move_to_end('a')                 # move 'a' to the end (most-recently-used)
print(list(od.items()))             # [('b',2),('c',3),('a',1)]

od.move_to_end('b', last=False)     # move 'b' to the front (least-recently-used)
print(list(od.items()))             # [('b',2),('c',3),('a',1)]

# popitem(last=True/False) — choose end
print(od.popitem(last=False))       # ('b', 2) — pops FROM FRONT


---
# 5. Pattern: existence / lookup

**Story:** the brute-force solution has a nested loop where the inner is "does X exist in the array?" Replace the inner scan with a set/dict lookup. O(n²) → O(n).

**Trigger phrases:** "have I seen this before?", "is the complement here?", "are they unique?"

**Variants:**
- Pure existence → `set`
- Existence + index/position → `dict[value] = index`
- Existence + count → `dict[value] = count`

In [ ]:
# 5.1 Two Sum (unsorted, return indices) — LC 1
def two_sum(nums, target):
    seen = {}                       # value -> index
    for i, v in enumerate(nums):
        if target - v in seen:
            return [seen[target - v], i]
        seen[v] = i
    return [-1, -1]

assert two_sum([2, 7, 11, 15], 9) == [0, 1]
assert two_sum([3, 2, 4], 6) == [1, 2]
print("two_sum OK")


In [ ]:
# 5.2 Contains duplicate — LC 217
def contains_duplicate(nums):
    return len(set(nums)) != len(nums)

# More explicit version showing the early-exit benefit
def contains_duplicate_v2(nums):
    seen = set()
    for v in nums:
        if v in seen:
            return True
        seen.add(v)
    return False

assert contains_duplicate([1, 2, 3, 1]) == True
assert contains_duplicate([1, 2, 3, 4]) == False
print("contains_duplicate OK")


In [ ]:
# 5.3 Contains duplicate within distance K — LC 219
def contains_nearby_duplicate(nums, k):
    seen = {}                       # value -> most recent index
    for i, v in enumerate(nums):
        if v in seen and i - seen[v] <= k:
            return True
        seen[v] = i
    return False

assert contains_nearby_duplicate([1, 2, 3, 1], 3) == True
assert contains_nearby_duplicate([1, 2, 3, 1, 2, 3], 2) == False
print("contains_nearby_duplicate OK")


In [ ]:
# 5.4 Happy number — LC 202 — cycle detection via set
def is_happy(n):
    seen = set()
    while n != 1 and n not in seen:
        seen.add(n)
        n = sum(int(d) ** 2 for d in str(n))
    return n == 1

assert is_happy(19) == True
assert is_happy(2) == False
print("is_happy OK")


**Practice — existence / lookup**

| Concept | LeetCode |
|---|---|
| Two Sum | LC 1 |
| Contains duplicate | LC 217 |
| Contains duplicate II (within k indices) | LC 219 |
| Happy number | LC 202 |
| Single number (XOR or set arithmetic) | LC 136 |
| Intersection of two arrays | LC 349 |
| Find common characters | LC 1002 |
| Missing number (set, math, or XOR) | LC 268 |


---
# 6. Pattern: frequency counting

**Story:** problems that ask about "how many times," "top K most frequent," "are these anagrams," or "first non-repeating character." Build a histogram in O(n) and answer the query in O(1) or O(k).

**Tool choice:**
- `Counter` — pythonic, has `most_common`, supports arithmetic
- `defaultdict(int)` — slightly faster for plain counting in hot loops
- **Fixed-size list of 26** — fastest when keys are lowercase letters

In [ ]:
# 6.1 Top K frequent elements — LC 347
from collections import Counter

def top_k_frequent(nums, k):
    return [v for v, _ in Counter(nums).most_common(k)]

print(top_k_frequent([1, 1, 1, 2, 2, 3], 2))            # [1, 2]
print(top_k_frequent([4,1,-1,2,-1,2,3], 2))             # [-1, 2]


In [ ]:
# 6.2 Top K frequent — BUCKET SORT version, O(n) (faster than Counter+heap for huge n)
def top_k_frequent_bucket(nums, k):
    n = len(nums)
    freq = Counter(nums)
    # buckets[i] = list of values whose freq is i  (max possible freq = n)
    buckets = [[] for _ in range(n + 1)]
    for v, f in freq.items():
        buckets[f].append(v)
    res = []
    for f in range(n, 0, -1):       # walk from highest freq down
        for v in buckets[f]:
            res.append(v)
            if len(res) == k:
                return res
    return res

print(top_k_frequent_bucket([1, 1, 1, 2, 2, 3], 2))     # [1, 2]


In [ ]:
# 6.3 Valid anagram — LC 242
def is_anagram(s, t):
    if len(s) != len(t):
        return False
    return Counter(s) == Counter(t)

# Faster version with fixed-size array (lowercase English only)
def is_anagram_fast(s, t):
    if len(s) != len(t):
        return False
    counts = [0] * 26
    for cs, ct in zip(s, t):
        counts[ord(cs) - ord('a')] += 1
        counts[ord(ct) - ord('a')] -= 1
    return all(c == 0 for c in counts)

assert is_anagram("anagram", "nagaram") == True
assert is_anagram_fast("rat", "car") == False
print("anagram OK")


In [ ]:
# 6.4 First non-repeating character — LC 387
def first_uniq_char(s):
    c = Counter(s)
    for i, ch in enumerate(s):
        if c[ch] == 1:
            return i
    return -1

assert first_uniq_char("leetcode") == 0
assert first_uniq_char("loveleetcode") == 2
assert first_uniq_char("aabb") == -1
print("first_uniq_char OK")


In [ ]:
# 6.5 More-than-n/k occurrences (Boyer-Moore generalized)
# Hashmap version: O(n) time, O(n) space. The Boyer-Moore generalization is O(nk) time, O(k) space.
def more_than_n_by_k(arr, k):
    n = len(arr)
    threshold = n // k
    return [v for v, c in Counter(arr).items() if c > threshold]

print(sorted(more_than_n_by_k([30, 10, 20, 30, 30, 10, 40, 30, 30], 2)))  # [30]
print(sorted(more_than_n_by_k([30, 10, 20, 20, 10, 20, 30, 30], 4)))      # [20, 30]


**Traps**
- Forgetting that `Counter[missing] == 0`, not a KeyError. Both useful and dangerous — don't accidentally rely on it equaling None.
- Equality between two Counters ignores zero counts, so `Counter({'a':1}) == Counter({'a':1, 'b':0})` is True.
- For pure speed in a hot loop with letter keys, a list of size 26 beats `Counter`. Mention this if asked about optimization.

**Practice — frequency counting**

| Concept | LeetCode |
|---|---|
| Valid anagram | LC 242 |
| Group anagrams | LC 49 |
| Top K frequent elements | LC 347 |
| Top K frequent words | LC 692 |
| First non-repeating character | LC 387 |
| Sort characters by frequency | LC 451 |
| Find all anagrams in a string | LC 438 |
| Ransom note | LC 383 |
| Majority element (n/2) | LC 169 |
| Majority element II (n/3) | LC 229 |


---
# 7. Pattern: prefix sum + hashmap

**Story:** This pattern is the answer to the question "count or find a **subarray** whose sum / property matches X — but the array has negatives, so sliding window doesn't work."

**Mental model:** the running prefix sum is your "signature so far." If you've seen the signature `prefix - target` before, then the slice between then and now sums to exactly `target`. The hashmap remembers what signatures you've seen and how many times.

**Why initialize with `{0: 1}`?** Because the subarray starting at index 0 has no "earlier prefix" — its left boundary is the imaginary empty prefix with sum 0. Without `{0: 1}`, you'd miss every subarray that starts at index 0.

In [ ]:
# 7.1 Subarray sum equals K — THE classic — LC 560
from collections import defaultdict

def subarray_sum(nums, k):
    counts = defaultdict(int)
    counts[0] = 1                   # the empty-prefix base case
    prefix = 0
    answer = 0
    for v in nums:
        prefix += v
        answer += counts[prefix - k]
        counts[prefix] += 1
    return answer

assert subarray_sum([1, 1, 1], 2) == 2
assert subarray_sum([1, 2, 3], 3) == 2
assert subarray_sum([1, -1, 0], 0) == 3
print("subarray_sum OK")


In [ ]:
# 7.2 EXISTENCE of a subarray with given sum (not the count) — uses a set
def has_subarray_with_sum(arr, target):
    seen_prefixes = {0}             # again, empty prefix
    prefix = 0
    for v in arr:
        prefix += v
        if (prefix - target) in seen_prefixes:
            return True
        seen_prefixes.add(prefix)
    return False

assert has_subarray_with_sum([5, 8, 6, 13, 3, -1], 22) == True
assert has_subarray_with_sum([15, 2, 8, 10, -5, -8, 6], 3) == True
assert has_subarray_with_sum([3, 1, 5, 6], 10) == False
print("has_subarray_with_sum OK")


In [ ]:
# 7.3 LONGEST subarray with given sum — store FIRST occurrence index
def longest_subarray_with_sum(arr, target):
    first_seen = {0: -1}            # prefix sum -> earliest index (-1 for empty prefix)
    prefix = 0
    best = 0
    for i, v in enumerate(arr):
        prefix += v
        if (prefix - target) in first_seen:
            best = max(best, i - first_seen[prefix - target])
        # ONLY record the first occurrence — keeps the subarray as long as possible
        if prefix not in first_seen:
            first_seen[prefix] = i
    return best

assert longest_subarray_with_sum([5, 8, -4, -4, 9, -2, 2], 0) == 3
assert longest_subarray_with_sum([3, 1, 0, 1, 8, 2, 3], 5) == 4
assert longest_subarray_with_sum([8, 3, 7], 15) == 0
print("longest_subarray_with_sum OK")


In [ ]:
# 7.4 Subarray sums divisible by K — LC 974
def subarrays_div_by_k(nums, k):
    counts = defaultdict(int)
    counts[0] = 1
    prefix = 0
    answer = 0
    for v in nums:
        prefix += v
        # Python's % already returns non-negative for negative numerators IF k > 0
        # but to be safe with negative results from some intermediate ops, use ((p % k) + k) % k
        r = prefix % k
        answer += counts[r]
        counts[r] += 1
    return answer

assert subarrays_div_by_k([4, 5, 0, -2, -3, 1], 5) == 7
print("subarrays_div_by_k OK")


In [ ]:
# 7.5 Contiguous array — equal 0s and 1s — LC 525
# Trick: replace 0 with -1. Now equal-0s-and-1s means subarray sum = 0.
def find_max_length(nums):
    first_seen = {0: -1}
    prefix = 0
    best = 0
    for i, v in enumerate(nums):
        prefix += 1 if v == 1 else -1
        if prefix in first_seen:
            best = max(best, i - first_seen[prefix])
        else:
            first_seen[prefix] = i
    return best

assert find_max_length([0, 1]) == 2
assert find_max_length([0, 1, 0]) == 2
assert find_max_length([0, 0, 1, 0, 0, 0, 1, 1]) == 6
print("find_max_length OK")


**Trap**: the difference between counting (`+= count[prefix - k]`) vs longest subarray (`max(best, i - first_seen[prefix - k])`) is which thing the hashmap stores.
- **Count problems** → `count[prefix] = how many times we've seen this prefix`
- **Longest / earliest problems** → `first_seen[prefix] = earliest index this prefix appeared`
- **Existence problems** → just a set of seen prefixes

**Practice — prefix sum + hashmap**

| Concept | LeetCode |
|---|---|
| Subarray sum equals K | LC 560 |
| Contiguous array (0s & 1s) | LC 525 |
| Continuous subarray sum (multiple of K) | LC 523 |
| Subarray sums divisible by K | LC 974 |
| Number of subarrays with bounded sum (variant) | LC 1248 — Nice subarrays |
| Path sum III (tree variant) | LC 437 |


---
# 8. Pattern: hashing for grouping

**Story:** problems where you partition items into buckets that share some property. The trick is choosing the right **canonical key** — a representation that's identical for everything that belongs in the same bucket.

**Canonical keys you'll see in interviews:**
- Sorted string → groups anagrams
- Tuple of 26 letter-counts → faster anagram key
- Tuple of row & column zero pattern → group matrix rows
- `tuple(sorted(...))` → any unordered collection
- Some hashed property of an object — careful with floats

In [ ]:
# 8.1 Group anagrams — LC 49 — sorted-string key
def group_anagrams_sorted_key(strs):
    groups = defaultdict(list)
    for s in strs:
        key = ''.join(sorted(s))    # O(k log k) per string
        groups[key].append(s)
    return list(groups.values())

print(group_anagrams_sorted_key(["eat","tea","tan","ate","nat","bat"]))


In [ ]:
# 8.2 Group anagrams — TUPLE-OF-COUNTS key  (O(k) per string, faster)
def group_anagrams_count_key(strs):
    groups = defaultdict(list)
    for s in strs:
        counts = [0] * 26
        for ch in s:
            counts[ord(ch) - ord('a')] += 1
        groups[tuple(counts)].append(s)     # tuple is hashable; list is not
    return list(groups.values())

print(group_anagrams_count_key(["eat","tea","tan","ate","nat","bat"]))


In [ ]:
# 8.3 Group shifted strings — LC 249 — encode pattern of differences
def group_strings(strs):
    def signature(s):
        if len(s) == 1:
            return ('',)            # singletons all share one bucket
        return tuple((ord(s[i]) - ord(s[i-1])) % 26 for i in range(1, len(s)))
    groups = defaultdict(list)
    for s in strs:
        groups[signature(s)].append(s)
    return list(groups.values())

print(group_strings(["abc","bcd","acef","xyz","az","ba","a","z"]))


**Practice — grouping by canonical key**

| Concept | LeetCode |
|---|---|
| Group anagrams | LC 49 |
| Group shifted strings | LC 249 |
| Find duplicate subtrees | LC 652 — serialize tree as key |
| Encode and decode strings | LC 271 (related — building canonical forms) |


---
# 9. Pattern: bidirectional / one-to-one mapping

**Story:** "is X a valid mapping of Y?" Isomorphic strings, word patterns, etc. The trick is checking BOTH directions with TWO hashmaps. Forgetting the second map is the #1 bug.

**Why two maps?** If you only check `a → b`, you'd accept `(a, b)` and `(c, b)` both mapping to `b`. That's many-to-one, not one-to-one.

In [ ]:
# 9.1 Isomorphic strings — LC 205
def is_isomorphic(s, t):
    if len(s) != len(t):
        return False
    s_to_t, t_to_s = {}, {}
    for cs, ct in zip(s, t):
        if cs in s_to_t and s_to_t[cs] != ct:
            return False
        if ct in t_to_s and t_to_s[ct] != cs:
            return False
        s_to_t[cs] = ct
        t_to_s[ct] = cs
    return True

assert is_isomorphic("egg", "add") == True
assert is_isomorphic("foo", "bar") == False
assert is_isomorphic("paper", "title") == True
assert is_isomorphic("badc", "baba") == False   # b→b, a→a, d→b — collision!
print("is_isomorphic OK")


In [ ]:
# 9.2 Word pattern — LC 290
def word_pattern(pattern, s):
    words = s.split()
    if len(pattern) != len(words):
        return False
    p_to_w, w_to_p = {}, {}
    for p, w in zip(pattern, words):
        if p in p_to_w and p_to_w[p] != w:
            return False
        if w in w_to_p and w_to_p[w] != p:
            return False
        p_to_w[p] = w
        w_to_p[w] = p
    return True

assert word_pattern("abba", "dog cat cat dog") == True
assert word_pattern("abba", "dog cat cat fish") == False
assert word_pattern("aaaa", "dog cat cat dog") == False
print("word_pattern OK")


**Practice — bidirectional mapping**

| Concept | LeetCode |
|---|---|
| Isomorphic strings | LC 205 |
| Word pattern | LC 290 |
| Find and replace pattern | LC 890 |
| Verifying an alien dictionary (related — order check) | LC 953 |


---
# 10. Pattern: set arithmetic

**Story:** problems that ask about intersection, union, or "consecutive run." Building a `set` from the input and using set operations or membership checks is dramatically simpler than sorting or running multi-pointer logic.

In [ ]:
# 10.1 Intersection of two arrays — LC 349 (unique) / LC 350 (with multiplicity)
def intersection_unique(a, b):
    return list(set(a) & set(b))

def intersection_with_multiplicity(a, b):
    ca = Counter(a)
    cb = Counter(b)
    # Counter & = element-wise min — exactly the multiset intersection
    return list((ca & cb).elements())

print(intersection_unique([1, 2, 2, 1], [2, 2]))                 # [2]
print(intersection_with_multiplicity([1, 2, 2, 1], [2, 2]))      # [2, 2]


In [ ]:
# 10.2 Longest consecutive sequence — LC 128 — THE marquee set problem
def longest_consecutive(nums):
    s = set(nums)                   # O(n) set construction
    best = 0
    for v in s:
        # only START counting if v-1 is missing — guarantees we count each run once
        # this is what makes the total complexity O(n) and NOT O(n^2)
        if v - 1 not in s:
            cur = v
            length = 1
            while cur + 1 in s:
                cur += 1
                length += 1
            best = max(best, length)
    return best

assert longest_consecutive([100, 4, 200, 1, 3, 2]) == 4
assert longest_consecutive([0,3,7,2,5,8,4,6,0,1]) == 9
assert longest_consecutive([]) == 0
print("longest_consecutive OK")


In [ ]:
# 10.3 Single number — LC 136 — set arithmetic OR XOR
def single_number_set(nums):
    return sum(set(nums)) * 2 - sum(nums)
    # Why: every element except one appears twice.
    # Set sum doubled = 2*(sum of distinct). Subtract actual sum -> the lonely element.

def single_number_xor(nums):
    # XOR is the canonical interview answer — O(1) space
    result = 0
    for v in nums:
        result ^= v
    return result

assert single_number_set([2,2,1]) == 1
assert single_number_xor([4,1,2,1,2]) == 4
print("single_number OK")


**Practice — set arithmetic**

| Concept | LeetCode |
|---|---|
| Intersection of two arrays | LC 349 |
| Intersection of two arrays II | LC 350 |
| Union of two arrays | (variant) |
| Longest consecutive sequence | LC 128 |
| Single number | LC 136 |
| Single number II / III | LC 137, LC 260 |
| Happy number | LC 202 |
| Distribute candies (set size cap) | LC 575 |


---
# 11. Pattern: hashmap inside a sliding window

**Story:** classic sliding-window problems where the "state" of the window is a frequency map. Common across strings, but worth knowing under hashing because the data structure carrying the state is the hashmap.

**Template (memorize):**
```
freq = {}
l = 0
best = 0
for r in range(n):
    add s[r] to freq
    while window violates condition:
        remove s[l] from freq
        l += 1
    update best from (r - l + 1)
```

In [ ]:
# 11.1 Longest substring without repeating characters — LC 3
def length_of_longest_unique(s):
    last_seen = {}                  # char -> most recent index
    l = 0
    best = 0
    for r, ch in enumerate(s):
        if ch in last_seen and last_seen[ch] >= l:
            l = last_seen[ch] + 1   # jump l past the previous occurrence
        last_seen[ch] = r
        best = max(best, r - l + 1)
    return best

assert length_of_longest_unique("abcabcbb") == 3
assert length_of_longest_unique("bbbbb") == 1
assert length_of_longest_unique("pwwkew") == 3
print("length_of_longest_unique OK")


In [ ]:
# 11.2 Longest substring with at most K distinct characters — LC 340
def longest_k_distinct(s, k):
    freq = defaultdict(int)
    l = 0
    best = 0
    for r, ch in enumerate(s):
        freq[ch] += 1
        while len(freq) > k:
            freq[s[l]] -= 1
            if freq[s[l]] == 0:
                del freq[s[l]]
            l += 1
        best = max(best, r - l + 1)
    return best

assert longest_k_distinct("eceba", 2) == 3
assert longest_k_distinct("aa", 1) == 2
print("longest_k_distinct OK")


In [ ]:
# 11.3 Find all anagrams in a string — LC 438 — fixed-window frequency match
def find_anagrams(s, p):
    if len(p) > len(s):
        return []
    p_count = Counter(p)
    w_count = Counter(s[:len(p)])
    res = []
    if w_count == p_count:
        res.append(0)
    for r in range(len(p), len(s)):
        w_count[s[r]] += 1
        w_count[s[r - len(p)]] -= 1
        if w_count[s[r - len(p)]] == 0:
            del w_count[s[r - len(p)]]
        if w_count == p_count:
            res.append(r - len(p) + 1)
    return res

print(find_anagrams("cbaebabacd", "abc"))   # [0, 6]
print(find_anagrams("abab", "ab"))          # [0, 1, 2]


In [ ]:
# 11.4 Minimum window substring — LC 76 — the hardest of the lot
def min_window(s, t):
    if not t or not s:
        return ""
    need = Counter(t)
    missing = len(t)                # how many chars (with multiplicity) still needed
    l = 0
    start = 0
    best = float('inf')
    for r, ch in enumerate(s):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        if missing == 0:
            # contract from the left while the window still covers t
            while need[s[l]] < 0:
                need[s[l]] += 1
                l += 1
            if r - l + 1 < best:
                best = r - l + 1
                start = l
            # release one needed char to keep searching
            need[s[l]] += 1
            missing += 1
            l += 1
    return "" if best == float('inf') else s[start:start + best]

assert min_window("ADOBECODEBANC", "ABC") == "BANC"
assert min_window("a", "a") == "a"
assert min_window("a", "aa") == ""
print("min_window OK")


**Practice — hashmap-in-window**

| Concept | LeetCode |
|---|---|
| Longest substring without repeating | LC 3 |
| Longest substring with K distinct | LC 340 |
| Fruit into baskets | LC 904 |
| Find all anagrams | LC 438 |
| Permutation in string | LC 567 |
| Minimum window substring | LC 76 |
| Longest repeating character replacement | LC 424 |
| Substring with concatenation of all words | LC 30 |


---
# 12. Pattern: design problems

**Story:** "design a data structure that supports..." These reward you for composing a hashmap with one more structure (list, doubly linked list, heap) to get O(1) on every operation.

**Canonical examples:**
- **LRU Cache** — `dict` + doubly linked list, or `OrderedDict` shortcut
- **RandomizedSet** — `dict` (val → index) + list (the data)
- **Twitter / News Feed** — `defaultdict(list)` + heap to merge posts
- **TimeMap** — `dict[key] = list of (timestamp, value)` + binary search

Knowing **why** OrderedDict makes LRU one-line easy is the most senior-sounding answer here.

In [ ]:
# 12.1 LRU Cache — LC 146 — the OrderedDict shortcut
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.cache = OrderedDict()      # key -> value, ordered by recency

    def get(self, key):
        if key not in self.cache:
            return -1
        self.cache.move_to_end(key)     # mark as most recently used — O(1)
        return self.cache[key]

    def put(self, key, value):
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.cap:
            self.cache.popitem(last=False)  # evict the LEAST recently used

c = LRUCache(2)
c.put(1, 1); c.put(2, 2)
print(c.get(1))             # 1
c.put(3, 3)                 # evicts key 2
print(c.get(2))             # -1
c.put(4, 4)                 # evicts key 1
print(c.get(1), c.get(3), c.get(4))     # -1 3 4


In [ ]:
# 12.2 Insert/Delete/GetRandom in O(1) — LC 380 — RandomizedSet
import random

class RandomizedSet:
    def __init__(self):
        self.vals = []                  # the data
        self.pos = {}                   # value -> its index in self.vals

    def insert(self, val):
        if val in self.pos:
            return False
        self.pos[val] = len(self.vals)
        self.vals.append(val)
        return True

    def remove(self, val):
        if val not in self.pos:
            return False
        # Trick: swap target with last element, pop last — O(1) instead of O(n)
        idx = self.pos[val]
        last = self.vals[-1]
        self.vals[idx] = last
        self.pos[last] = idx
        self.vals.pop()
        del self.pos[val]
        return True

    def getRandom(self):
        return random.choice(self.vals)

s = RandomizedSet()
print(s.insert(1), s.insert(2), s.remove(1), s.insert(3))    # True True True True
print(s.getRandom() in {2, 3})                                # True


In [ ]:
# 12.3 Time-based key-value store — LC 981 — hashmap + sorted list + binary search
from bisect import bisect_right

class TimeMap:
    def __init__(self):
        self.store = defaultdict(list)    # key -> list of (timestamp, value), ascending by ts

    def set(self, key, value, timestamp):
        self.store[key].append((timestamp, value))

    def get(self, key, timestamp):
        arr = self.store.get(key)
        if not arr:
            return ""
        # find rightmost entry with ts <= timestamp
        i = bisect_right(arr, (timestamp, chr(127))) - 1   # tuple compare hack
        return arr[i][1] if i >= 0 else ""

tm = TimeMap()
tm.set("foo", "bar", 1)
print(tm.get("foo", 1))     # bar
print(tm.get("foo", 3))     # bar
tm.set("foo", "bar2", 4)
print(tm.get("foo", 4))     # bar2
print(tm.get("foo", 5))     # bar2


**Why these problems matter:** they show you can **compose** data structures. The interviewer is looking for "I'll use a dict to get O(1) lookup, and a [list / linked list / heap] to get [random access / order / top-k]."

**Practice — design**

| Concept | LeetCode |
|---|---|
| LRU Cache | LC 146 |
| LFU Cache | LC 460 |
| Insert/Delete/GetRandom O(1) | LC 380 |
| Insert/Delete/GetRandom — Duplicates allowed | LC 381 |
| Time-based key-value store | LC 981 |
| Design HashMap (from scratch) | LC 706 |
| Design HashSet (from scratch) | LC 705 |
| Design Twitter | LC 355 |
| Two Sum III - Data structure | LC 170 |
| Logger Rate Limiter | LC 359 |


---
# 13. Hashable keys — beyond strings & ints

**Story:** real-world problems often need **compound keys** — (x, y) coordinates, (row, col) of a matrix, an unordered group of items, a custom object. The rule: the key must be **immutable and hashable**.

The escape hatches when an obvious-looking key isn't hashable:
- `list` → use `tuple(...)`
- `set` → use `frozenset(...)`
- `dict` → use `tuple(sorted(d.items()))`
- Custom class → implement `__hash__` and `__eq__` together

In [ ]:
# 13.1 Tuple keys — grid coordinates
visited = set()
visited.add((0, 0))
visited.add((1, 2))
print((0, 0) in visited)        # True

# Counting paired observations
pairs = Counter()
pairs[(1, 2)] += 1
pairs[(1, 2)] += 1
pairs[(2, 1)] += 1
print(pairs)


In [ ]:
# 13.2 Frozenset keys — group anagrams alternative (only when chars unique!) /
#                       store sets of items where order shouldn't matter
seen_groups = set()
seen_groups.add(frozenset({'apple', 'banana'}))
seen_groups.add(frozenset({'banana', 'apple'}))   # same set — no double add
print(len(seen_groups))                            # 1

# Frozenset is hashable -> can sit inside another set or be a dict key
score = {frozenset({'A', 'B'}): 10, frozenset({'B', 'C'}): 7}
print(score[frozenset({'B', 'A'})])


In [ ]:
# 13.3 Custom class — must implement __hash__ AND __eq__ together
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __eq__(self, other):
        return isinstance(other, Point) and self.x == other.x and self.y == other.y
    def __hash__(self):
        return hash((self.x, self.y))           # delegate to tuple hash
    def __repr__(self):
        return f"Point({self.x},{self.y})"

s = set()
s.add(Point(0, 0))
s.add(Point(0, 0))                              # treated as duplicate
print(s)
print(Point(0, 0) in s)


**The hash-eq contract** (interviewers love this):
- If `a == b`, then `hash(a) == hash(b)` — MUST hold.
- The reverse need not hold (collisions are fine).
- If you override `__eq__` without `__hash__`, Python makes the class unhashable by default. Always pair them.

**Practice — hashable keys**

| Concept | LeetCode |
|---|---|
| Number of islands (visited set with tuple keys) | LC 200 |
| Walls and gates / Rotting oranges (grid + visited) | LC 286 / LC 994 |
| Find duplicate subtrees (serialize as key) | LC 652 |
| Sentence similarity II (union-find or graph with set keys) | LC 737 |


---
# 14. Pattern-recognition cheat sheet

| Problem signal | Reach for |
|---|---|
| "Have I seen X before?" | `set` |
| "Two sum / pair sum / complement search" | `dict[value] = index` |
| "How many times does X appear?" | `Counter` or `defaultdict(int)` |
| "Group by some canonical key (anagrams, shifted strings)" | `defaultdict(list)` keyed on canonical form |
| "Subarray sum equals K" (negatives allowed, count) | Prefix sum + `defaultdict(int)`, seeded with `{0: 1}` |
| "Longest subarray with sum K" / earliest match | Prefix sum + `dict` of FIRST occurrence, seeded with `{0: -1}` |
| "Existence of subarray with sum K" | Prefix sum + `set` |
| "Equal 0s and 1s" / "equal As and Bs" | Map values to +1/-1, then prefix-sum-equals-0 |
| "Is X a one-to-one mapping of Y?" | TWO hashmaps (forward & reverse) |
| "Top K frequent" | `Counter.most_common(k)` or bucket sort |
| "Longest consecutive sequence" | `set(arr)` + only-start-at-run-beginning trick |
| "Window with frequency constraint" | Sliding window + `Counter` / `defaultdict(int)` |
| "Random insert/delete/get in O(1)" | `dict` (val→idx) + `list` (data), swap-with-last trick |
| "LRU / recency-aware cache" | `OrderedDict` + `move_to_end` |
| "Compound key (coordinates, unordered group)" | `tuple` or `frozenset` as the key |

---

# 15. What to say out loud in the interview

When you reach for a hashmap, narrate **three things** so the interviewer knows you understand it:

1. **Why** — "A nested loop would be O(n²). If I memoize what I've seen so far in a hashmap, the inner lookup becomes O(1) and I drop to O(n)."
2. **What goes in it** — "I'll store value → index" or "previous prefix sum → count of times seen."
3. **Why it's safe** — "Lookup is O(1) on average. Worst case is O(n) per op for adversarial inputs, but Python uses a randomized hash so we don't worry about that here."

When asked "**why O(1)?**" be precise:
> "Hashmaps are O(1) average for insert, lookup, and delete because we compute `hash(key) % capacity` to find the slot directly. The constant assumes a low load factor — CPython resizes when the table is about 2/3 full, which keeps collisions rare. Worst case is O(n) if every key hashes to the same bucket, but this is essentially impossible on real inputs."

When asked "**dict vs OrderedDict?**" in 2026:
> "Regular `dict` preserves insertion order since Python 3.7, so for most purposes `dict` is enough. `OrderedDict` is still useful for two things: it has order-sensitive equality, and it has `move_to_end` and `popitem(last=False)`, which makes implementing an LRU cache trivial."

When asked "**why can't a list be a key?**":
> "A dict key must be hashable, and `__hash__` and `__eq__` must be consistent for the lifetime of the key. Lists are mutable — you could append to a list after using it as a key, and now the bucket the dict put it in is wrong. Tuples of immutable items are the standard workaround."

---

# 16. Common bugs in hashing problems

1. **Forgetting `{0: 1}` (count) or `{0: -1}` (longest) in prefix-sum-hashmap problems.** You'll miss every subarray starting at index 0.
2. **Storing the WRONG side of the mapping for prefix sums.** Counting uses `count[prefix]`; longest uses `first_seen[prefix]`. Mix them up and you'll either over-count or shorten your answer.
3. **Only one mapping in isomorphic-string problems.** You need both directions or you'll miss many-to-one collisions.
4. **Accessing a `defaultdict` to check existence.** `dd[k]` creates `k` with the default value as a side effect. Use `k in dd` for existence checks.
5. **Using a `list` as a dict key.** TypeError. Convert to `tuple`.
6. **Comparing `OrderedDict`s expecting order to be ignored.** The whole point of OrderedDict is that order is part of equality.
7. **`set.remove` vs `set.discard`.** `remove` raises KeyError if missing; `discard` silently does nothing. In defensive code, prefer `discard`.
8. **Counter equality treating missing keys as zero.** `Counter({'a':1}) == Counter({'a':1, 'b':0})` is True. Usually a feature, sometimes a footgun.
9. **Float keys.** `0.1 + 0.2 != 0.3` — never use floats as keys unless you're sure of the source. Round first or convert to a representation.
